### Transform Payments Data
 1. Extract Date and Time from Payments_TimeStamp and create a new columns Payment_date and Payment_time
 2. Map Payment_status to contain descriptive values
    (1-Success, 2-Pending, 3-Cancelled, 4-Failed)
 3. Write transformed data to the Silver schema

In [0]:
dfPayments = spark.table("gizmobox_sivan.bronze.payments");
display(dfPayments)

###1.Extract Date and Time from payment_timestamp
[document for date_format function](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/date_format)

In [0]:
# cast(date_format(payment_timestamp,'yyyy-MM-dd') as date) as payment_date,
#      date_format(payment_timestamp,'HH:mm:ss') as payment_time,
from pyspark.sql.functions import date_format
dfPaymentsAfterCasting = (
    dfPayments.select(
            dfPayments.payment_id.cast("integer"), 
            dfPayments.order_id.cast("integer"), 
            dfPayments.payment_timestamp.cast("timestamp"), 
            date_format(dfPayments.payment_timestamp,'yyyy-MM-dd').cast("date").alias("payment_date"),
            date_format(dfPayments.payment_timestamp,'HH:mm:ss').alias("payment_time"),
            dfPayments.payment_status.cast("integer"), 
            dfPayments.payment_method.cast("string"))
)

display(dfPaymentsAfterCasting)

###2. Map payment_status to contain descriptive values
(1-Success, 2-Pending, 3-Cancelled, 4-Failed)

[Pyspark When Function](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.when.html)

In [0]:
from pyspark.sql.functions import when
df_mapped_payments = (
        dfPaymentsAfterCasting
        .select(
            'payment_id',
            'order_id',
            'payment_date',
            'payment_time',
            when(dfPaymentsAfterCasting.payment_status == 1, 'Success')
            .when(dfPaymentsAfterCasting.payment_status == 2, 'Pending')
            .when(dfPaymentsAfterCasting.payment_status == 3, 'Cancelled')
            .when(dfPaymentsAfterCasting.payment_status == 4, 'Failed')
            .alias('payment_status'),
            'payment_method')
        )

display(df_mapped_payments)

### 3. Write transformed data to Silver schema

In [0]:
df_mapped_payments.writeTo("gizmobox_sivan.silver.py_payments").createOrReplace()

In [0]:
%sql
select * from gizmobox_sivan.silver.py_payments